# Pretrain SimMIM + VICReg from MoBY (contrastive self-supervised) weights

Self-supervised pretraining of a Swin-Tiny encoder with SimMIM (masked
reconstruction) + VICReg (variance / invariance / covariance) on
multi-channel synaptic microscopy patches.  The encoder is initialised
from **MoBY** (Xie et al., 2021) — a hybrid of MoCo v2 + BYOL that
learns view-invariant representations via a momentum encoder, key
queue and contrastive loss, pre-trained on ImageNet-1K for 300 epochs.

### Why MoBY?

Self-supervised contrastive features tend to transfer better across
domain gaps than supervised classification features.  MoBY trains
the encoder to be *invariant* to augmentations (crop, color, blur)
without class labels, so the learned representations focus on
structural/textural similarity rather than semantic categories —
more relevant for fluorescence microscopy where ImageNet classes
are meaningless.

### Hyperparameter rationale (paper references)

Since we are transferring from one SSL method (MoBY / contrastive)
to another (SimMIM + VICReg), the training schedule is tuned for
**gentle continuation pre-training** rather than aggressive adaptation:

- **base_lr = 1.0e-4**: SimMIM (Xie et al. 2022, App. A.1) uses peak
  LR 8e-4 at BS=2048 for 100ep Swin-B pretraining.  Sqrt-scaled to
  BS=32 this gives exactly 1.0e-4.  Lower than the timm-ImageNet
  variant (1.5e-4) — MoBY features are already self-supervised, so
  we *refine*, not overwrite.
- **layer_decay = 0.85**: stronger LLRD than timm init (0.90).
  MoBY's contrastive pretraining trains all stages end-to-end;
  early-layer features should be protected.  SimMIM fine-tuning
  uses 0.7–0.9 (Xie et al. 2022, Table 8).
- **warmup_epochs = 15**: the objective changes from contrastive
  to masked reconstruction — a longer warmup lets the optimizer
  adapt to the new loss landscape.
- **freeze_encoder_epochs = 2**: heads (decoder, projector) start
  random; two frozen epochs let them catch up before encoder
  gradients propagate.
- **weight_decay = 0.05**: standard across Swin, SimMIM, VICReg,
  and MoBY papers.

### Checkpoint download

Download the **MoBY Swin-T 300ep** checkpoint from Google Drive:

```
https://drive.google.com/file/d/1PS1Q0tAnUfBWLRPxh9iUrinAxeq7Y--u/view
```

(From https://github.com/SwinTransformer/Transformer-SSL)

**⚠️ WARNING:** The Transformer-SSL README has **swapped download
links** between the DeiT-S and Swin-T rows.  The GitHub link named
``moby_swin_t_300ep_pretrained.pth`` is actually DeiT-Small.  Use
the **Google Drive link** above for the correct Swin-T checkpoint.

Place it under ``data/checkpoints/`` and set ``pretrained_ckpt_path``
below.

All knobs live in the **Configuration** section below.  Everything
downstream is data-driven; no logic branches on hard-coded paths.


## Imports


In [ ]:
import os, sys, json, time
from datetime import datetime
from pathlib import Path


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt


## Path setup


In [ ]:
NB_DIR = Path.cwd().resolve()
NB_NEW = NB_DIR.parent if NB_DIR.parent.name == 'notebooks' else NB_DIR.parents[1]
ROOT   = NB_NEW.parent
for p in (NB_NEW, ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
print('NB_NEW:', NB_NEW)
print('ROOT  :', ROOT)


In [ ]:
import nb_utils as U
from nb_utils import (
    BaseCfg, DataCfg, ModelCfg, TrainCfg, SSLCfg, dump_config,
    seed_everything, setup_logger, CSVMetricLogger,
    build_dataloaders, compute_channel_stats,
    MicroscopyTwoViewTransform, ValSingleViewTransform,
    build_swin_encoder, build_simmim_vicreg_heads, count_params,
    load_pretrained_into_encoder,
    random_block_mask, apply_mask,
    compute_simmim_vicreg_loss, validation_simmim,
    param_groups_layer_decay, make_warmup_cosine,
    save_checkpoint, load_checkpoint, find_latest_checkpoint,
    SANITY_TRAIN_INDICES, SANITY_VAL_INDICES,
    fixed_two_view_batch, fixed_single_view_batch, overfit_on_batch,
    plot_two_views, plot_channel_histograms, plot_recon_panel,
    plot_loss_curves, plot_overfit_curves, plot_embedding_2d,
    extract_pooled_embeddings, effective_rank, mean_pairwise_cos, reduce_2d,
    build_run_label, eval_recon_batch, reload_best_checkpoint,
    post_training_reconstruction, plot_post_training_curves,
    embedding_diagnostics,
)

## Configuration

Key differences from the timm-ImageNet variant:

| knob | timm-ImageNet | **MoBY** | reason |
|------|:---:|:---:|--------|
| `init_source` | `timm_imagenet` | **`moby`** | contrastive SSL |
| `base_lr` | 1.5e-4 | **1.0e-4** | MoBY features already SSL-trained |
| `warmup_epochs` | 10 | **15** | objective-switch needs gentler ramp |
| `layer_decay` | 0.90 | **0.85** | protect early MoBY contrastive features |
| `freeze_encoder_epochs` | 1 | **2** | random heads catch up to strong encoder |


In [ ]:
base_cfg = BaseCfg(
    seed             = 42,
    output_root      = '../outputs',
    experiment_name  = 'simmim_vicreg_pretrain',
    tag              = 'moby',
    method_name      = 'simmim_vicreg',
    init_source      = 'moby',
    timm_model_name  = 'swin_tiny_patch4_window7_224',  # unused for moby, kept for config parity
    pretrained_ckpt_path = '../../../data/checkpoints/moby_swin_t_300ep.pth',
    resume_path      = None,             # set to a .pt file or run dir to resume
    dry_run          = False,
)


In [ ]:
data_cfg = DataCfg(
    data_root        = '../../../data/patches_128',
    exclude_patterns = ['KONTROLA'],
    val_split        = 0.10,
    batch_size       = 32,
    num_workers      = 2,
    pin_memory       = True,
    channel_names    = ['pre_synaptic', 'post_synaptic', 'structural'],
)


In [ ]:
model_cfg = ModelCfg(
    in_channels    = 3,
    img_size       = 128,
    feature_size   = 96,
    patch_size     = 4,
    window_size    = 7,
    depths         = (2, 2, 6, 2),
    num_heads      = (3, 6, 12, 24),
    dropout_path_rate = 0.05,
)


In [ ]:
train_cfg = TrainCfg(
    epochs                = 200,
    warmup_epochs         = 15,
    base_lr               = 1.0e-4,
    head_lr               = 5.0e-4,
    weight_decay          = 0.05,
    layer_decay           = 0.85,
    grad_clip_norm        = 5.0,
    freeze_encoder_epochs = 2,
    save_every_n_epochs   = 25,
    val_metric_key        = 'ssl_loss',
    val_metric_direction  = 'min',
    encoder_save_name     = 'pretrained_encoder_simmim_vicreg_moby.pt',
)


In [ ]:
ssl_cfg = SSLCfg(
    mask_ratio       = 0.60,
    mask_block_size  = 16,
    loss_kind        = 'l1',
    lambda_sim       = 25.0,
    lambda_std       = 25.0,
    lambda_cov       = 1.0,
    w_recon          = 1.0,
    w_vicreg         = 1.0,
    projector_hidden = 256,
    projector_dim    = 256,
)


## Output directory and logger


In [ ]:
RUN_TS = datetime.now().strftime('%Y%m%d_%H%M%S')
save_dir = Path(base_cfg.output_root) / f'{base_cfg.experiment_name}_{base_cfg.tag}_{RUN_TS}'
save_dir.mkdir(parents=True, exist_ok=True)
print('save_dir =', save_dir)


In [ ]:
logger = setup_logger('nb', save_dir / 'run.log')
logger.info(f'experiment = {base_cfg.experiment_name}')
logger.info(f'tag        = {base_cfg.tag}')
logger.info(f'method     = {base_cfg.method_name}')
logger.info(f'init_source= {base_cfg.init_source}')
logger.info(f'save_dir   = {save_dir}')


In [ ]:
dump_config(
    save_dir / 'config.json',
    base=base_cfg, data=data_cfg, model=model_cfg, train=train_cfg, ssl=ssl_cfg,
)
logger.info('config.json written')


## Seed and device


In [ ]:
generator = seed_everything(base_cfg.seed)
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f'seed   = {base_cfg.seed}')
logger.info(f'device = {device}')
if device.type == 'cuda':
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    logger.info(f'gpu    = {name}  ({mem:.1f} GB)')


## Data

In [ ]:
# Raw dataset
from data_utils.patch_dataset import PatchDataset
raw_dataset = PatchDataset(root=data_cfg.data_root, exclude_patterns=data_cfg.exclude_patterns)
logger.info(f'raw patches = {len(raw_dataset)}')
sample = raw_dataset[0]
logger.info(f'sample shape = {tuple(sample.shape)}  dtype = {sample.dtype}')
assert sample.ndim == 3 and sample.shape[0] == model_cfg.in_channels
assert sample.shape[-1] == model_cfg.img_size


In [ ]:
# Channel stats (computed on the train split, pre-augmentation)

from torch.utils.data import random_split
n_val   = int(len(raw_dataset) * data_cfg.val_split)
n_train = len(raw_dataset) - n_val
train_subset, val_subset = random_split(raw_dataset, [n_train, n_val], generator=generator)
logger.info(f'split: train={n_train}  val={n_val}')

ch_mean, ch_std = compute_channel_stats(
    train_subset, in_channels=model_cfg.in_channels,
    max_samples=data_cfg.channel_stats_max_samples,
)
for name, m, s in zip(data_cfg.channel_names, ch_mean.tolist(), ch_std.tolist()):
    logger.info(f'  ch[{name:>14s}] mean={m:.5f}  std={s:.5f}')
stats = {'channel_names': list(data_cfg.channel_names), 'mean': ch_mean.tolist(), 'std': ch_std.tolist()}
(save_dir / 'channel_stats.json').write_text(json.dumps(stats, indent=2))


In [ ]:
# Augmentation pipelines
train_transform = MicroscopyTwoViewTransform(ch_mean, ch_std)
val_transform   = ValSingleViewTransform(ch_mean, ch_std)


In [ ]:
# DataLoaders
from nb_utils.data import TransformedSubset
train_ds = TransformedSubset(train_subset, train_transform)
val_ds   = TransformedSubset(val_subset,   val_transform)
common = dict(
    num_workers=data_cfg.num_workers,
    pin_memory=data_cfg.pin_memory,
    persistent_workers=data_cfg.num_workers > 0,
    prefetch_factor=2 if data_cfg.num_workers > 0 else None,
)
train_loader = DataLoader(train_ds, batch_size=data_cfg.batch_size, shuffle=True,  drop_last=True, **common)
val_loader   = DataLoader(val_ds,   batch_size=data_cfg.batch_size, shuffle=False, **common)
logger.info(f'train batches = {len(train_loader)}  val batches = {len(val_loader)}')


## Sanity checks

In [ ]:
# Build encoder
encoder = build_swin_encoder(model_cfg).to(device)
logger.info(f'encoder params = {count_params(encoder) / 1e6:.2f} M')
# Load MoBY pretrained weights
load_summary = load_pretrained_into_encoder(encoder, base_cfg, model_cfg, logger=logger)
# Verify high coverage: MoBY Swin-T should load ~95%+ of encoder keys
n_loaded = load_summary['n_loaded']
n_target = load_summary['n_target_params']
pct = 100.0 * n_loaded / max(n_target, 1)
logger.info(f'MoBY coverage: {n_loaded}/{n_target} params loaded ({pct:.1f}%)')
if load_summary.get('n_shape_mismatch', 0) > 0:
    logger.warning(f'  shape mismatches: {load_summary["shape_mismatch"]}')
if n_loaded < 0.8 * n_target:
    raise RuntimeError(
            f'Only {pct:.1f}% of encoder loaded from MoBY — expected >=80%. '
        f'Check that the checkpoint is MoBY Swin-T (not DeiT-S — README links are swapped!).'
    )


In [ ]:
# Build heads (decoder + mask token + projector)
heads = build_simmim_vicreg_heads(model_cfg, ssl_cfg)
heads = {k: v.to(device) for k, v in heads.items()}
logger.info(f'head params = {count_params(heads) / 1e6:.2f} M')


In [ ]:
# Sanity check: dataloader yields a two-view pair of the right shape
batch = next(iter(train_loader))
assert isinstance(batch, (list, tuple)) and len(batch) == 2, batch
v1, v2 = batch
assert v1.shape == v2.shape
assert v1.shape[1:] == (model_cfg.in_channels, model_cfg.img_size, model_cfg.img_size)
assert v1.dtype == torch.float32
assert torch.isfinite(v1).all() and torch.isfinite(v2).all()
diff = (v1 - v2).abs().mean().item()
assert diff > 1e-3, f'view1 == view2  (mean abs diff = {diff:.2e})'
logger.info(f'[ok] views shape={tuple(v1.shape)}  v1-v2 diff={diff:.4f}')


In [ ]:
# Sanity check: two-view augmentation visualisation (canonical indices)
_ = plot_two_views(
    raw_subset=train_subset,
    transform=train_transform,
    indices=SANITY_TRAIN_INDICES,
    channel_names=data_cfg.channel_names,
    suptitle='Sanity batch -- two-view augmentation',
    save_to=save_dir / 'sanity_two_views.png',
)
plt.show()


In [ ]:
# Sanity check: post-normalisation channel histograms
_ = plot_channel_histograms(
    v1, channel_names=data_cfg.channel_names,
    save_to=save_dir / 'sanity_histograms.png',
)
plt.show()


In [ ]:
# Sanity check: encoder forward + per-stage shapes
encoder.eval()
with torch.no_grad():
    feats = encoder(v1[:2].to(device).contiguous())
assert isinstance(feats, list) and len(feats) == 5
for i, f in enumerate(feats):
    logger.info(f'  stage {i}: {tuple(f.shape)}')
encoder.train()


In [ ]:
# Sanity check: gradient actually flows through encoder + heads
encoder.train()
for h in heads.values(): h.train()
_opt = torch.optim.AdamW(
    list(encoder.parameters())
    + [p for h in heads.values() for p in h.parameters()],
    lr=1e-4,
)
_v1 = v1.to(device); _v2 = v2.to(device)
_opt.zero_grad(set_to_none=True)
_loss, _m = compute_simmim_vicreg_loss(encoder, heads, _v1, _v2, ssl_cfg)
_loss.backward()
n_grad = sum(1 for p in encoder.parameters() if p.grad is not None)
n_tot  = sum(1 for _ in encoder.parameters())
_opt.step()
del _opt, _loss, _m, _v1, _v2
logger.info(f'[ok] grad flow: {n_grad}/{n_tot} encoder params got gradients')


## Overfit check

In [ ]:
# Build optimizer + scheduler + GradScaler

encoder_groups = param_groups_layer_decay(
    encoder,
    base_lr=train_cfg.base_lr,
    weight_decay=train_cfg.weight_decay,
    layer_decay=train_cfg.layer_decay,
)
head_params = [p for h in heads.values() for p in h.parameters()]
head_group  = {'params': head_params, 'lr': train_cfg.head_lr, 'weight_decay': train_cfg.weight_decay}
optimizer = torch.optim.AdamW(encoder_groups + [head_group], betas=(0.9, 0.999))
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, make_warmup_cosine(train_cfg.warmup_epochs, train_cfg.epochs),
)
scaler = torch.amp.GradScaler(device.type, enabled=device.type == 'cuda')


### Sanity overfit on the canonical train batch

Trains the *real* model on the fixed ``SANITY_TRAIN_INDICES`` for a
small number of steps with a frozen mask. Per the project convention,
the encoder + heads are NOT auto-restored afterwards -- the warm-up
carries into the full training run. Set ``RESET_AFTER_SANITY=True``
if you want a byte-identical pre-overfit state.


In [ ]:
RUN_OVERFIT      = True
RESET_AFTER_SANITY = False  # user preference: do NOT auto-reset
N_OVERFIT_STEPS  = 200
OVERFIT_LR       = 3e-4


In [ ]:
x1_fixed, x2_fixed, sanity_idx = fixed_two_view_batch(
    train_subset, SANITY_TRAIN_INDICES, train_transform,
    seed=base_cfg.seed + 17,
)
x1_fixed = x1_fixed.to(device); x2_fixed = x2_fixed.to(device)
logger.info(f'sanity batch indices = {sanity_idx}  shape = {tuple(x1_fixed.shape)}')


In [ ]:
torch.manual_seed(base_cfg.seed + 18)
mask_fixed = random_block_mask(x1_fixed, ssl_cfg.mask_block_size, ssl_cfg.mask_ratio).clone()
logger.info(f'fixed mask coverage = {mask_fixed.mean().item():.3f}')


In [ ]:
if RUN_OVERFIT:
    overfit_result = overfit_on_batch(
        encoder, heads, x1_fixed, x2_fixed, ssl_cfg,
        n_steps=N_OVERFIT_STEPS, lr=OVERFIT_LR,
        grad_clip=train_cfg.grad_clip_norm, fixed_mask=mask_fixed,
        restore_state=RESET_AFTER_SANITY, logger=logger,
    )
else:
    overfit_result = None
    logger.info('overfit sanity skipped (RUN_OVERFIT=False)')


### Plot


In [ ]:
# Overfit-on-batch loss curves
if overfit_result is not None:
    _ = plot_overfit_curves(
        overfit_result['history'],
        suptitle=f'Sanity overfit on indices {sanity_idx}',
        save_to=save_dir / 'sanity_curves_overfit.png',
        run_label=build_run_label(base_cfg, extra='overfit'),
    )
    plt.show()

In [ ]:
# Overfit-on-batch reconstruction on the canonical batch
if overfit_result is not None:
    rec, v_masked = eval_recon_batch(encoder, heads, x1_fixed, mask_fixed)
    _ = plot_recon_panel(
        x1_fixed, v_masked, rec, mask_fixed, sanity_idx,
        ch_mean=ch_mean, ch_std=ch_std,
        suptitle='Sanity overfit -- reconstruction on canonical batch',
        save_to=save_dir / 'sanity_recon_overfit.png',
        run_label=build_run_label(base_cfg, extra='overfit'),
        channel_names=data_cfg.channel_names,
    )
    plt.show()
    encoder.train()
    for h in heads.values():
        h.train()

## Full training

### Reset for full training

The sanity overfit above mutates the encoder, the heads, and the
AdamW state. Treat it as a separate run -- rebuild every training
structure here so the full run starts from a clean slate:
encoder weights (MoBY per ``base_cfg``), heads,
optimizer, LR scheduler, ``GradScaler`` and the start_epoch /
best_val_metric counters. Resume logic is re-applied so it points
at a previous *full-training* checkpoint, not the overfit warm-up.


In [ ]:
# Re-seed so the full run is deterministic and independent of the overfit warm-up.
generator = seed_everything(base_cfg.seed)

# Rebuild encoder (re-applies ``base_cfg.init_source``: moby).
encoder = build_swin_encoder(model_cfg).to(device)
load_summary = load_pretrained_into_encoder(encoder, base_cfg, model_cfg, logger=logger)
logger.info(f'[reset] encoder params = {count_params(encoder) / 1e6:.2f} M')

# Rebuild heads (decoder + mask token + projector).
heads = build_simmim_vicreg_heads(model_cfg, ssl_cfg)
heads = {k: v.to(device) for k, v in heads.items()}
logger.info(f'[reset] head params    = {count_params(heads) / 1e6:.2f} M')

# Rebuild optimizer + scheduler + GradScaler.
encoder_groups = param_groups_layer_decay(
    encoder,
    base_lr=train_cfg.base_lr,
    weight_decay=train_cfg.weight_decay,
    layer_decay=train_cfg.layer_decay,
)
head_params = [p for h in heads.values() for p in h.parameters()]
head_group  = {'params': head_params, 'lr': train_cfg.head_lr, 'weight_decay': train_cfg.weight_decay}
optimizer = torch.optim.AdamW(encoder_groups + [head_group], betas=(0.9, 0.999))
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, make_warmup_cosine(train_cfg.warmup_epochs, train_cfg.epochs),
)
scaler = torch.amp.GradScaler(device.type, enabled=device.type == 'cuda')

# Reset training-state counters.
start_epoch     = 1
best_val_metric = float('inf') if train_cfg.val_metric_direction == 'min' else float('-inf')
best_epoch      = 0

# Re-apply resume logic so it targets a previous full-training checkpoint,
# never the overfit warm-up state.
_resume = base_cfg.resume_path
if _resume is not None:
    p = Path(_resume)
    if p.is_dir():
        p = find_latest_checkpoint(p)
    if p is None or not Path(p).exists():
        logger.warning(f'[reset] resume path {_resume!r} not found -- starting fresh')
    else:
        ckpt = load_checkpoint(
            p, encoder=encoder, heads=heads,
            optimizer=optimizer, scheduler=scheduler, scaler=scaler,
            map_location=device,
        )
        start_epoch     = int(ckpt.get('epoch', 0)) + 1
        best_val_metric = ckpt.get('val_metric', best_val_metric) or best_val_metric
        best_epoch      = int(ckpt.get('epoch', 0))
        logger.info(f'[reset] resumed from {p} (epoch {ckpt.get("epoch")})')
logger.info(f'[reset] start_epoch = {start_epoch}  best_val_metric = {best_val_metric}')


### Resume from a previous interrupted session

Set ``base_cfg.resume_path`` to a checkpoint file (``last.pt``,
``best_model.pt``, ...) or to a directory -- in which case the most
recent checkpoint under it is auto-selected. Leave it ``None`` for a
fresh run.


In [ ]:
start_epoch     = 1
best_val_metric = float('inf') if train_cfg.val_metric_direction == 'min' else float('-inf')
best_epoch      = 0

_resume = base_cfg.resume_path
if _resume is not None:
    p = Path(_resume)
    if p.is_dir():
        p = find_latest_checkpoint(p)
    if p is None or not Path(p).exists():
        logger.warning(f'resume path {_resume!r} not found -- starting fresh')
    else:
        ckpt = load_checkpoint(
            p, encoder=encoder, heads=heads,
            optimizer=optimizer, scheduler=scheduler, scaler=scaler,
            map_location=device,
        )
        start_epoch     = int(ckpt.get('epoch', 0)) + 1
        best_val_metric = ckpt.get('val_metric', best_val_metric) or best_val_metric
        best_epoch      = int(ckpt.get('epoch', 0))
        logger.info(f'resumed from {p} (epoch {ckpt.get("epoch")})')
logger.info(f'start_epoch = {start_epoch}  best_val_metric = {best_val_metric}')


### CSV metric logger


In [ ]:
csv_fields = [
    'epoch', 'phase',
    'train_loss', 'val_metric',
    'lr_encoder', 'lr_head',
    'epoch_time_s', 'train_time_s', 'val_time_s',
    'best_val_metric', 'best_epoch',
    'grad_norm_mean', 'grad_norm_max',
    'train_recon', 'train_sim', 'train_std', 'train_cov', 'train_vicreg',
    'val_recon', 'val_std', 'val_cov',
]
csv_logger = CSVMetricLogger(save_dir / 'metrics.csv', csv_fields)


### Training-loop helpers


In [ ]:
def freeze_encoder(enc, freeze):
    for p in enc.parameters():
        p.requires_grad = not freeze

def set_train(enc, hds, training):
    enc.train(training)
    for h in hds.values():
        h.train(training)

def heads_iter_lrs(opt):
    enc_lrs  = [g['lr'] for g in opt.param_groups if 'stage' in g]
    head_lrs = [g['lr'] for g in opt.param_groups if 'stage' not in g]
    return (max(enc_lrs) if enc_lrs else 0.0,
            head_lrs[0] if head_lrs else 0.0)

def all_trainable_params(enc, hds):
    return ([p for p in enc.parameters() if p.requires_grad]
            + [p for h in hds.values() for p in h.parameters() if p.requires_grad])


### Full training loop


In [ ]:
train_losses, val_history, lr_history = [], [], []
total_t0 = time.time()


In [ ]:
_better = (lambda new, best: new < best) if train_cfg.val_metric_direction == 'min' else (lambda new, best: new > best)

if base_cfg.dry_run:
    logger.info('dry_run=True -- skipping full training loop')
else:
    for epoch in range(start_epoch, train_cfg.epochs + 1):
        in_warmup = epoch <= train_cfg.freeze_encoder_epochs
        freeze_encoder(encoder, in_warmup)
        phase = 'frozen' if in_warmup else 'full'

        set_train(encoder, heads, True)
        running, grads = 0.0, []
        comp_accum = {}
        t_train = time.time()
        pbar = tqdm(train_loader, desc=f'ep {epoch}/{train_cfg.epochs} [{phase}]', leave=False)
        for v1_b, v2_b in pbar:
            v1_b = v1_b.to(device, non_blocking=True)
            v2_b = v2_b.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device.type, enabled=device.type == 'cuda'):
                loss, _m = compute_simmim_vicreg_loss(encoder, heads, v1_b, v2_b, ssl_cfg)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            gn = torch.nn.utils.clip_grad_norm_(
                all_trainable_params(encoder, heads),
                max_norm=train_cfg.grad_clip_norm,
            )
            grads.append(gn.item())
            scaler.step(optimizer)
            scaler.update()
            running += loss.item()
            for _k, _v in _m.items():
                comp_accum[_k] = comp_accum.get(_k, 0.0) + float(_v)
            pbar.set_postfix(loss=f'{loss.item():.4f}')
        n_tb = max(1, len(train_loader))
        train_loss = running / n_tb
        train_components = {k: v / n_tb for k, v in comp_accum.items()}
        train_time = time.time() - t_train
        scheduler.step()

        set_train(encoder, heads, False)
        val_accum = {}
        n_vb = 0
        t_val = time.time()
        with torch.no_grad():
            for v_b in val_loader:
                if isinstance(v_b, (list, tuple)):
                    v_b = v_b[0]
                v_b = v_b.to(device, non_blocking=True)
                vm = validation_simmim(encoder, heads, v_b, ssl_cfg)
                for k, v in vm.items():
                    val_accum[k] = val_accum.get(k, 0.0) + float(v)
                n_vb += 1
            for k in val_accum:
                val_accum[k] /= max(1, n_vb)
        val_time   = time.time() - t_val
        val_metric = val_accum[train_cfg.val_metric_key]

        enc_lr, head_lr = heads_iter_lrs(optimizer)
        epoch_time = train_time + val_time
        gmean = sum(grads) / max(1, len(grads))
        gmax  = max(grads) if grads else 0.0

        improved = ''
        if _better(val_metric, best_val_metric):
            best_val_metric = val_metric
            best_epoch      = epoch
            improved        = ' *best*'
            save_checkpoint(
                save_dir / 'best_model.pt',
                encoder=encoder, heads=heads,
                optimizer=optimizer, scheduler=scheduler, scaler=scaler,
                epoch=epoch, val_metric=val_metric, train_loss=train_loss,
                extra={'all_val_metrics': val_accum,
                       'channel_mean': ch_mean.tolist(),
                       'channel_std':  ch_std.tolist()},
            )
        save_checkpoint(
            save_dir / 'last.pt',
            encoder=encoder, heads=heads,
            optimizer=optimizer, scheduler=scheduler, scaler=scaler,
            epoch=epoch, val_metric=val_metric, train_loss=train_loss,
            extra={'channel_mean': ch_mean.tolist(),
                   'channel_std':  ch_std.tolist()},
        )
        if train_cfg.save_every_n_epochs and epoch % train_cfg.save_every_n_epochs == 0:
            save_checkpoint(
                save_dir / f'epoch_{epoch:04d}.pt',
                encoder=encoder, heads=heads,
                optimizer=optimizer, scheduler=scheduler, scaler=scaler,
                epoch=epoch, val_metric=val_metric, train_loss=train_loss,
                extra={'channel_mean': ch_mean.tolist(),
                       'channel_std':  ch_std.tolist()},
            )

        train_losses.append(train_loss)
        val_history.append(val_accum)
        lr_history.append((enc_lr, head_lr))
        csv_logger.log(dict(
            epoch=epoch, phase=phase,
            train_loss=train_loss, val_metric=val_metric,
            lr_encoder=enc_lr, lr_head=head_lr,
            epoch_time_s=epoch_time, train_time_s=train_time, val_time_s=val_time,
            best_val_metric=best_val_metric, best_epoch=best_epoch,
            grad_norm_mean=gmean, grad_norm_max=gmax,
            train_recon=train_components.get('recon'),
            train_sim=train_components.get('sim'),
            train_std=train_components.get('std'),
            train_cov=train_components.get('cov'),
            train_vicreg=train_components.get('vicreg'),
            val_recon=val_accum.get('recon'),
            val_std=val_accum.get('std'),
            val_cov=val_accum.get('cov'),
        ))
        logger.info(
            f'ep {epoch:3d}/{train_cfg.epochs} [{phase}]  '
            f'train={train_loss:.5f}  val[{train_cfg.val_metric_key}]={val_metric:.5f}  '
            f'recon(t/v)={train_components.get("recon",0):.4f}/{val_accum.get("recon",0):.4f}  '
            f'std(t/v)={train_components.get("std",0):.4f}/{val_accum.get("std",0):.4f}  '
            f'lr(enc/head)={enc_lr:.2e}/{head_lr:.2e}  t={epoch_time:.1f}s  gn={gmean:.2f}{improved}'
        )
    csv_logger.close()
    logger.info(f'total training time = {(time.time() - total_t0) / 60:.1f} min')


## After training plotting

In [ ]:
# Reload best checkpoint and build a run-identifying label for the figures.
ckpt = reload_best_checkpoint(save_dir, encoder, heads, device, logger=logger)
run_label = build_run_label(
    base_cfg,
    epoch=(ckpt or {}).get('epoch'),
    val_metric=(ckpt or {}).get('val_metric'),
)

In [ ]:
# Reconstruction on the canonical TRAIN sanity batch
_ = post_training_reconstruction(
    encoder, heads, train_subset, train_transform, ssl_cfg,
    ch_mean=ch_mean, ch_std=ch_std, save_dir=save_dir,
    base_seed=base_cfg.seed, view='train',
    run_label=run_label, channel_names=data_cfg.channel_names,
    device=device,
)

In [ ]:
# Reconstruction on the canonical VAL sanity patches (no aug)
_ = post_training_reconstruction(
    encoder, heads, val_subset, val_transform, ssl_cfg,
    ch_mean=ch_mean, ch_std=ch_std, save_dir=save_dir,
    base_seed=base_cfg.seed, view='val',
    run_label=run_label, channel_names=data_cfg.channel_names,
    device=device,
)

In [ ]:
# Training curves (total / SimMIM recon / VICReg components / LR schedule)
_ = plot_post_training_curves(save_dir, run_label=run_label, logger=logger)

In [ ]:
# Embedding diagnostics on the val set (effective rank, mean pairwise cos)
_ = embedding_diagnostics(
    encoder, val_loader, device, save_dir,
    base_seed=base_cfg.seed, run_label=run_label, logger=logger,
)

## Save encoder-only checkpoint


In [ ]:
if best_path.exists():
    src = torch.load(best_path, map_location=device, weights_only=False)
    enc_path = save_dir / train_cfg.encoder_save_name
    torch.save({
        'encoder_state_dict': src['encoder_state_dict'],
        'channel_mean':       src.get('channel_mean', ch_mean.tolist()),
        'channel_std':        src.get('channel_std',  ch_std.tolist()),
        'epoch':              src.get('epoch'),
        'val_metric':         src.get('val_metric'),
        'init_source':        base_cfg.init_source,
        'method_name':        base_cfg.method_name,
    }, enc_path)
    logger.info(f'encoder saved to {enc_path}')
else:
    logger.warning('no best_model.pt -- nothing to export')
